## Allstate Insurance Premium Feature Analysis

As part of the group project for UW Green Bay's Artificial Intelligence course, we investigated a dataset released as part of an investigation surrounding certain insurance premium pricing practices of Allstate's auto policies in Maryland. The purpose of this Jupyter notebook is to perform an initial investigation into the dataset, as well as note which input features would be worth exploring to predict the output premium for the given dataset.

As a sidenote, while we were completing the exploration of this dataset, we quickly realized one key factor that affects the actual value of the premium was omitted – namely, the vehicle value itself. Without this information, we either drastically underfit the data by only considering a person's age, gender, and zipcode; or we overfit the data by considering the proposed premium prior to customer grouping adjustments (which was part of the investigation, and why this data was released to begin with). For this reason, we only decided to use this dataset for examination. Although it *did* provide real-world data that we could look at, it did not provide us *all the information we needed* to predict a premium with reasonable accuracy. Because we could not find an alternative real-world dataset including this necessary information that was also freely available, we needed to leverage a separate synthetic dataset to predict the actual premium itself. (This exploration is contained in a separate Jupyter notebook.)

As is the case with any project, the first step is to import the necessary dependencies. In this case, I leveraged the Scikit-learn Linear Regression, Support Vector Machines (Regression), Decision Trees (Regression), and Random Forest (Regression). I also imported the necessary dependencies to split the packages into training and testing datasets, scale the input data, and evaluate the model performance after training was finished.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelBinarizer

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

Then, the next step was to import the CSV files included in the dataset. To do this, we leveraged a `base_dir` variable containing all CSV files for the given dataset, which we leveraged to open each file with Python string formatting. The result can be seen below:

In [2]:
base_dir = './data_sources/allstate_investigation_dataset'
cgr_definitions = pd.read_csv(f'{base_dir}/cgr_definitions.csv')
cgr_premiums = pd.read_csv(f'{base_dir}/cgr_premiums.csv')
maryland_demographics = pd.read_csv(f'{base_dir}/maryland_demographics.csv')
maryland_economics = pd.read_csv(f'{base_dir}/maryland_economics.csv')
territory_definitions = pd.read_csv(f'{base_dir}/territory_definitions.csv')

In [3]:
cgr_definitions.head()

,cgr,aa,bb,cc,va,dd,hh,ss
0,1OW,0.1066,0.1066,0.1066,0.1066,0.1066,0.1066,0.1066
1,7HB,0.1071,0.1071,0.1071,0.1071,0.1071,0.1071,0.1071
2,1OE,0.1076,0.1076,0.1076,0.1076,0.1076,0.1076,0.1076
3,1OD,0.1082,0.1082,0.1082,0.1082,0.1082,0.1082,0.1082
4,I9Y,0.1087,0.1087,0.1087,0.1087,0.1087,0.1087,0.1087


In [4]:
cgr_premiums.head()

,territory,gender,birthdate,ypc,current_premium,indicated_premium,selected_premium,underlying_premium,fixed_expenses,underlying_total_premium,cgr_factor,cgr
0,601,M,10/5/1947,0,863.97,830.58,862.57,673.06,175.98,849.04,1.02,ZHK
1,601,F,7/6/1953,0,828.63,611.14,826.43,612.75,175.98,788.73,1.06,6NS
2,601,M,4/18/1956,0,1000.59,593.99,996.60,858.20,175.98,1034.18,0.96,Z2D
3,601,F,8/16/1956,0,700.42,547.95,697.84,571.49,180.48,751.97,0.91,D7G
4,601,F,1/23/1957,0,505.92,448.33,504.56,333.71,152.08,485.79,1.06,3YN


In [5]:
maryland_demographics.head()

,GEO.id,GEO.id2,GEO.display-label,HC01_VC03,HC02_VC03,HC03_VC03,HC04_VC03,HC01_VC04,HC02_VC04,HC03_VC04,...,HC03_VC108,HC04_VC108,HC01_VC109,HC02_VC109,HC03_VC109,HC04_VC109,HC01_VC110,HC02_VC110,HC03_VC110,HC04_VC110
0,8600000US20601,20601,ZCTA5 20601,25028,908,25028,NaN,12190,603,48.7,...,18131,NaN,8746,415,48.2,1.3,9385,416,51.8,1.3
1,8600000US20602,20602,ZCTA5 20602,26708,940,26708,NaN,12250,620,45.9,...,18233,NaN,7998,479,43.9,1.6,10235,455,56.1,1.6
2,8600000US20603,20603,ZCTA5 20603,31784,1134,31784,NaN,14660,630,46.1,...,21664,NaN,9720,523,44.9,1.4,11944,505,55.1,1.4
3,8600000US20606,20606,ZCTA5 20606,443,167,443,NaN,214,97,48.3,...,332,NaN,165,68,49.7,12.5,167,71,50.3,12.5
4,8600000US20607,20607,ZCTA5 20607,10639,729,10639,NaN,5099,447,47.9,...,7936,NaN,3729,334,47.0,2.6,4207,306,53.0,2.6


In [6]:
maryland_economics.head()

,GEO.id,GEO.id2,GEO.display-label,HC01_VC03,HC02_VC03,HC03_VC03,HC04_VC03,HC01_VC04,HC02_VC04,HC03_VC04,...,HC03_VC178,HC04_VC178,HC01_VC179,HC02_VC179,HC03_VC179,HC04_VC179,HC01_VC180,HC02_VC180,HC03_VC180,HC04_VC180
0,8600000US20601,20601,ZCTA5 20601,19753,688,19753,NaN,14478,555,73.3,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2
1,8600000US20602,20602,ZCTA5 20602,20338,719,20338,NaN,14896,607,73.2,...,12.6,6.9,NaN,NaN,12.7,4.1,NaN,NaN,15.3,4.2
2,8600000US20603,20603,ZCTA5 20603,23434,849,23434,NaN,17829,786,76.1,...,3.5,4.4,NaN,NaN,5.0,2.3,NaN,NaN,9.0,4.1
3,8600000US20606,20606,ZCTA5 20606,371,128,371,NaN,157,83,42.3,...,7.5,11.8,NaN,NaN,16.1,18.3,NaN,NaN,43.7,35.2
4,8600000US20607,20607,ZCTA5 20607,8323,523,8323,NaN,6174,500,74.2,...,1.3,1.5,NaN,NaN,0.7,0.7,NaN,NaN,8.6,7.9


In [7]:
territory_definitions.head()

,county,county_code,territory,zipcode,town,area
0,ALLEGANY,1,1502,21502,CRESAPTOWN,160
1,ALLEGANY,1,1502,21502,CUMBERLAND,160
2,ALLEGANY,1,1521,21521,BARTON,160
3,ALLEGANY,1,1524,21524,CORRIGANVILLE,160
4,ALLEGANY,1,1529,21529,ELLERSLIE,160


The next step for us was to take each imported CSV file and combine it into a single Pandas DataFrame. To do this, we looked at each dataset and noted which attribtes were in common that we could leverage as primary keys.

While many were straightforward (having the primary key with the same column name in each DataFrame), others of them had different column names in each DataFrame (such as the `zipcode` and `GEO.id2` fields in the `territory_definitions` and the `maryland_demographics` DataFrames respectively). These required special joining techniques using the `left_on` and `right_on` parameters, but most leveraged a simple `on` as the joining parameter.

Finally, both the `maryland_demographics` and the `maryland_economics` DataFrames had the same column names (although the data itself likely represented different attributes), so I had to use the `suffixes` parameter to tell which data attribute was originally imported from which DataFrame. After doing a little digging, it became clear that many of these duplicate attributes were likely economic data derived from census data and had little to do with actual insurance analysis itself (but were included in the dataset as part of the Allstate investigation, as this was key to the pricing objective of the company).

In [8]:
dataset = cgr_premiums.merge(cgr_definitions, on='cgr', how='inner')
dataset = dataset.merge(territory_definitions, on='territory', how='inner')
dataset = dataset.merge(maryland_demographics, left_on='zipcode', right_on='GEO.id2', how='inner')
dataset = dataset.merge(maryland_economics, on='GEO.id2', how='inner', suffixes=('_demo', '_econ'))

dataset.head()

,territory,gender,birthdate,ypc,current_premium,indicated_premium,selected_premium,underlying_premium,fixed_expenses,underlying_total_premium,...,HC03_VC178,HC04_VC178,HC01_VC179,HC02_VC179,HC03_VC179,HC04_VC179,HC01_VC180,HC02_VC180,HC03_VC180,HC04_VC180
0,601,M,10/5/1947,0,863.97,830.58,862.57,673.06,175.98,849.04,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2
1,601,M,10/5/1947,0,863.97,830.58,862.57,673.06,175.98,849.04,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2
2,601,F,7/6/1953,0,828.63,611.14,826.43,612.75,175.98,788.73,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2
3,601,F,7/6/1953,0,828.63,611.14,826.43,612.75,175.98,788.73,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2
4,601,M,4/18/1956,0,1000.59,593.99,996.60,858.20,175.98,1034.18,...,7.5,3.6,NaN,NaN,4.0,1.7,NaN,NaN,14.9,4.2


Finally, we had a single DataFrame that we could work with. However, since there were many columns in this combined dataset, we found it helpful to print out each one. However, it is worth noting that we only worked with and performed feature analysis on a select few of these that were likely to be relevant to actual premium prediction.

In [9]:
for col in dataset.columns:
    print(col, end=', ')

territory, gender, birthdate, ypc, current_premium, indicated_premium, selected_premium, underlying_premium, fixed_expenses, underlying_total_premium, cgr_factor, cgr, aa, bb, cc, va, dd, hh, ss, county, county_code, zipcode, town, area, GEO.id_demo, GEO.id2, GEO.display-label_demo, HC01_VC03_demo, HC02_VC03_demo, HC03_VC03_demo, HC04_VC03_demo, HC01_VC04_demo, HC02_VC04_demo, HC03_VC04_demo, HC04_VC04_demo, HC01_VC05_demo, HC02_VC05_demo, HC03_VC05_demo, HC04_VC05_demo, HC01_VC08_demo, HC02_VC08_demo, HC03_VC08_demo, HC04_VC08_demo, HC01_VC09_demo, HC02_VC09_demo, HC03_VC09_demo, HC04_VC09_demo, HC01_VC10, HC02_VC10, HC03_VC10, HC04_VC10, HC01_VC11_demo, HC02_VC11_demo, HC03_VC11_demo, HC04_VC11_demo, HC01_VC12_demo, HC02_VC12_demo, HC03_VC12_demo, HC04_VC12_demo, HC01_VC13, HC02_VC13, HC03_VC13, HC04_VC13, HC01_VC14_demo, HC02_VC14_demo, HC03_VC14_demo, HC04_VC14_demo, HC01_VC15_demo, HC02_VC15_demo, HC03_VC15_demo, HC04_VC15_demo, HC01_VC16_demo, HC02_VC16_demo, HC03_VC16_demo, HC04

Finally, once we looked at all the columns in the dataset, we decided which ones were worth investigating as part of our project. We decided that the `gender`, `birthdate`, and `zipcode` were obvious input features for the investigation. However, we also discovered that there was no feature containing the value of the item itself being insured. With this in mind, it would be virtually impossible to predict the actual value of the premium itself using these features alone – as most policies cover collision and comprehensive (coverages on the value of the vehicle itself) rather than liability only (coverages based mostly on the individual driving the vehicle).

However, something that we *could* do on this real-world dataset was performing feature analysis on the dataset using Decision Tree and Random Forest regression techniques. To do this, we essentially took the input features we mentioned before (those related to the individual) and combined them with the fixed expenses and premiums *other* than what the customer was actually paying (the ones predicted internally via Allstate's proprietary algorithm). We selected the output feature as the premium that the customer was actually paying and performed a feature analysis on this correlation.

In [10]:
features_of_interest = dataset[['gender', 'birthdate', 'zipcode', 'underlying_premium', 'underlying_total_premium', 'cgr_factor', 'selected_premium', 'indicated_premium', 'fixed_expenses', 'current_premium']].dropna()

features_of_interest['birthdate'] = pd.to_datetime(features_of_interest['birthdate'])
features_of_interest['age'] = pd.Timestamp.now() - features_of_interest['birthdate']

features_of_interest = features_of_interest.drop('birthdate', axis=1)

Before we could actually start training models with the data, we needed to scale and standardize it to spec. To do this, we leveraged the `LabelBinarizer` and `StandardScalar` to scale categorical and continuous data respectively. The strategy of looping through the columns to perform this data manipulation is shown below:

In [11]:
labelBinaryColumns = ['gender', 'zipcode']
standardScalarColumns = ['age', 'underlying_premium', 'underlying_total_premium', 'cgr_factor', 'selected_premium', 'indicated_premium', 'fixed_expenses', 'current_premium']

for column in labelBinaryColumns:
    labelBinary = LabelBinarizer()
    features_of_interest[column] = labelBinary.fit_transform(features_of_interest[column])

for column in standardScalarColumns:
    standardScalar = StandardScaler()
    features_of_interest[column] = standardScalar.fit_transform(features_of_interest[column].values.reshape(-1, 1))

X = features_of_interest[['gender', 'age', 'zipcode', 'cgr_factor', 'fixed_expenses', 'underlying_premium', 'underlying_total_premium', 'selected_premium', 'indicated_premium']]
y = features_of_interest[['current_premium']]

# X = X[0:500]
# y = y[0:500]

### Linear Regression – A Basic Example

One of the most basic regression techniques taught in UW Green Bay's online Artificial Intelligence course is Linear Regression. Since the input variables contained premiums generated by AllState and these input features were highly correlated with what the company actually charged customers (the output variable), the evaluation metrics provided very strong results.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [13]:
y_pred = linear_regression.predict(X_test)

In [14]:
print('Mean Squared Error (MSE):', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))
print('Mean Absolute Error (MAE):', mean_absolute_error(y_test, y_pred))

Mean Squared Error (MSE): 0.005232370293646531
R2 Score: 0.994904912710523
Mean Absolute Error (MAE): 0.03865588518948911


### Support Vector Machines – Stronger Data Manipulation

We played around with various kernels for the AllState dataset. However, depending on which input features we were including, we would get varying results. If we included only the features of the individual itself while ignoring AllState's proprietary premiums, we would drastically underfit the model and get maybe 50% accuracy at most when using highly tuned models. On the other hand, we would get almost-perfect accuracy when we included AllState's predicted premiums as input features with the output feature as what the customer is actually paying, which is very insignificant data to say the least. However, our analysis can be seen below:

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
svr = SVR(kernel='rbf')
svr.fit(X_train, np.ravel(y_train))

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [16]:
y_pred = svr.predict(X_test)

In [17]:
print('Mean Squared Error (MSE):', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))
print('Mean Absolute Error (MAE):', mean_absolute_error(y_test, y_pred))

Mean Squared Error (MSE): 0.009584188972383776
R2 Score: 0.9906672737836553
Mean Absolute Error (MAE): 0.03893393341255373


### Decision Tree and Random Forest – The Power of Feature Analysis

While we could not predict the actual premium itself using any significant input features, we were able to see which input features were stronger predictors of the customer's actual premium. To achieve this, we leveraged the Decision Tree and Random Forest techniques as can be seen below:

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
decision_tree = DecisionTreeRegressor(random_state=42)
decision_tree.fit(X_train, np.ravel(y_train))

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_l

In [19]:
y_pred = decision_tree.predict(X_test)

In [20]:
print('Mean Squared Error (MSE):', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))
print('Mean Absolute Error (MAE):', mean_absolute_error(y_test, y_pred))

Mean Squared Error (MSE): 0.001701793152596772
R2 Score: 0.9983428572187172
Mean Absolute Error (MAE): 0.005422184829657277


In [21]:
pd.DataFrame(
    data={
        'feature': decision_tree.feature_names_in_,
        'importance': decision_tree.feature_importances_,
    }
).sort_values('importance', ascending=False)

,feature,importance
7,selected_premium,9.809133e-01
6,underlying_total_premium,1.481045e-02
8,indicated_premium,3.731080e-03
5,underlying_premium,1.860022e-04
1,age,1.305698e-04
3,cgr_factor,1.154813e-04
4,fixed_expenses,8.451036e-05
0,gender,2.858381e-05
2,zipcode,1.670697e-08


Although Random Forest is generally more-accurate (especially on more-complex datasets), we got similar output results for Random Forest too. However, one thing that differed was the extreme of the differences between the feature importances (as can be seen below) which is likely caused, in part, by random forest providing multiple estimators to us.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
random_forest = RandomForestRegressor(random_state=42, n_estimators=50)
random_forest.fit(X_train, np.ravel(y_train))

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples

In [23]:
y_pred = decision_tree.predict(X_test)

In [24]:
pd.DataFrame(
    data={
        'feature': random_forest.feature_names_in_,
        'importance': random_forest.feature_importances_,
    }
).sort_values('importance', ascending=False)

,feature,importance
7,selected_premium,9.765415e-01
6,underlying_total_premium,1.713006e-02
8,indicated_premium,3.366999e-03
5,underlying_premium,2.517635e-03
1,age,1.700828e-04
3,cgr_factor,1.331335e-04
4,fixed_expenses,1.195528e-04
0,gender,2.061037e-05
2,zipcode,3.962758e-07
